In [ ]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

# Authenticatation
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = "" 
    login(token=HF_TOKEN)
    print("Authenticated via explicit string.")

PyTorch 2.8.0+cu128 | Detected 1 GPUs.


Authenticated via explicit string.


In [9]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL plus the raw counts needed to pool correctly across the
        whole corpus. bpb/bpc/bpw below are still returned for per-sentence inspection,
        but the CORPUS-LEVEL (headline) numbers must be computed by pooling nll and the
        *_count fields across all rows -- never by averaging these per-row ratios."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            seq_len = inputs["input_ids"].shape[1]
            n_scored = seq_len - 1          # tokens actually predicted -- needed for token PPL
            neg_log_likelihood = outputs.loss.item() * n_scored

        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb, "bpc": bpc, "bpw": bpw,
            "n_scored": n_scored,
            "n_chars": chars, "n_bytes": bytes_count, "n_words": words,
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


In [10]:
diverse_sentences_path = "/home/jovyan/projects/NLP Research/diverse_sentences.csv"
sinhala_mixed_path = "/home/jovyan/projects/NLP Research/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


In [13]:
# The root cause is that `torchaudio` is already cached as `None` in `sys.modules` (from a prior failed import attempt). `setdefault` does not overwrite an existing `None` entry, so the stub never takes effect. The fix is to force-assign it directly.

import sys
import types
sys.modules['torchaudio'] = types.ModuleType('torchaudio')

model_id = "microsoft/phi-4"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN,trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 across both GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
model.eval()

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Device Map: {model.hf_device_map}")

Loading microsoft/phi-4...


model.safetensors.index.json:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded.
Precision: torch.float16 | Device Map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 0, 'model.layers.26': 0, 'model.layers.27': 'cpu', 'model.layers.28': 'cpu', 'model.layers.29': 'cpu', 'model.layers.30': 'cpu', 'model.layers.31': 'cpu', 'model.layers.32': 'cpu', 'model.layers.33': 'cpu', 'model.layers.34': 'cpu', 'model.layers.35': 'cpu', 'model.layers.36': 'cpu', 'model.layers.37': 'cpu', 'model.layers.38': 'cpu', 'model.layers.39': 'cpu',

In [14]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    pilot_sample = diverse_df.head(20)

    for index, row in pilot_sample.iterrows():
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

Starting Pilot Run (First 20 items)...


Processed 5/20 parallel pairs...


Processed 10/20 parallel pairs...


Processed 15/20 parallel pairs...


Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,unicode_nscored,unicode_nchars,unicode_nbytes,unicode_nwords,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw,romanized_nscored,romanized_nchars,romanized_nbytes,romanized_nwords
0,0,67.698637,1.252160,3.255616,16.278081,53,30,78,6,83.329638,3.163665,3.163665,20.036543,13,38,38,6
1,1,54.348104,1.187996,3.266989,19.601935,44,24,66,4,67.729318,3.489741,3.489741,24.428188,9,28,28,4
2,2,63.787177,1.108740,2.968563,15.337574,54,31,83,6,88.796099,3.558491,3.558491,21.350949,13,36,36,6


In [ ]:
import csv
import pandas as pd
from tqdm import tqdm

full_diverse_output = 'phi4_14b_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

full_mixed_output = 'phi4_14b_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw',
        'mixed_nscored', 'mixed_nchars', 'mixed_nbytes', 'mixed_nwords'
    ])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw'],
            mix_metrics['n_scored'], mix_metrics['n_chars'], mix_metrics['n_bytes'], mix_metrics['n_words']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")


# --- Corrected aggregation: pooled (corpus-level) BPB/BPC/BPW + token-level Perplexity ---
def calculate_full_aggregates(csv_path, model_label="Phi-4-14B"):
    df = pd.read_csv(csv_path)
    ln2 = math.log(2)

    def pooled_stats(prefix):
        nll_sum = df[f'{prefix}_nll'].sum()
        return {
            'bpb': nll_sum / (df[f'{prefix}_nbytes'].sum() * ln2),
            'bpc': nll_sum / (df[f'{prefix}_nchars'].sum() * ln2),
            'bpw': nll_sum / (df[f'{prefix}_nwords'].sum() * ln2),
            # this is the number to compare against the paper's Table:
            'ppl_token': math.exp(nll_sum / df[f'{prefix}_nscored'].sum()),
        }

    uni = pooled_stats('unicode')
    rom = pooled_stats('romanized')

    results_df = pd.DataFrame({
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)',
                   'Token-Level Perplexity'],
        'Unicode (Native)': [uni['bpb'], uni['bpc'], uni['bpw'], uni['ppl_token']],
        'Romanized (Singlish)': [rom['bpb'], rom['bpc'], rom['bpw'], rom['ppl_token']],
    })
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY (corpus-pooled)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala (corpus-pooled)").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

In [ ]:
def calculate_mixed_aggregates(csv_path, model_label="Phi-4-14B"):
    df = pd.read_csv(csv_path)
    ln2 = math.log(2)
    nll_sum = df['mixed_nll'].sum()

    pooled_bpb = nll_sum / (df['mixed_nbytes'].sum() * ln2)
    pooled_bpc = nll_sum / (df['mixed_nchars'].sum() * ln2)
    pooled_bpw = nll_sum / (df['mixed_nwords'].sum() * ln2)
    pooled_ppl = math.exp(nll_sum / df['mixed_nscored'].sum())

    # per-sentence values, used ONLY for the dispersion column below -- not the headline numbers above
    per_sentence_ppl = (df['mixed_nll'] / df['mixed_nscored']).apply(math.exp)

    results_df = pd.DataFrame({
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)',
                   'Token-Level Perplexity'],
        'Mixed-Script (corpus-pooled)': [pooled_bpb, pooled_bpc, pooled_bpw, pooled_ppl],
        'Per-sentence Std Dev': [df['mixed_bpb'].std(), df['mixed_bpc'].std(),
                                 df['mixed_bpw'].std(), per_sentence_ppl.std()],
    })

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script (corpus-pooled)': '{:.4f}',
        'Per-sentence Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)